# Differential expression analysis


You have run the nf-core/rnaseq pipeline and checked the first quality control metrics of your fastq files. This was, however, only the primary analysis and we want to take it further.

Due to the computational demand of the pipeline, you only ran the pipeline on two of the 16 samples in the study yesterday. We provide you an essential output of nf-core/rnaseq pipeline in the `data` folder: It contains the combined epression matrix as produced by Salmon, which provides transcript levels for each gene (rows) and each sample (columns).


We would now like to understand exactly the difference between the expression in our groups of mice. 
Which pipeline would you use for this?

nf-core/differentialabundance

Have a close look at the pipeline's "Usage" page on the [nf-core docs](nf-co.re). You will need to create a samplesheet (based on the column names in the provided matrix).

In [1]:
import glob
import pandas as pd

matrix_path = glob.glob("data/salmon.merged.gene_count*.tsv")[0]
print(matrix_path)

mat = pd.read_csv(matrix_path, sep="\t", index_col=0)
print(mat.shape)
print(mat.columns.tolist())      # sample names -> needed for the samplesheet

contrasts = pd.read_csv("data/contrasts.csv")
contrasts

data/salmon.merged.gene_counts.tsv
(45706, 17)
['gene_name', 'Sham_oxy_1', 'Sham_oxy_2', 'Sham_oxy_3', 'Sham_oxy_4', 'Sham_Sal_1', 'Sham_Sal_2', 'Sham_Sal_3', 'Sham_Sal_4', 'SNI_oxy_1', 'SNI_oxy_2', 'SNI_oxy_3', 'SNI_oxy_4', 'SNI_Sal_1', 'SNI_Sal_2', 'SNI_Sal_3', 'SNI_Sal_4']


,id,variable,reference,target
0,condition_control_treated,Condition,SNI_Sal,SNI_oxy
1,condition_control_treated_test,Condition,Sham_Sal,Sham_oxy


In [16]:
# create sample sheet
import pandas as pd

samplesheet = pd.DataFrame({
    "sample": [
        "Sham_oxy_1",
        "Sham_oxy_2",
        "Sham_oxy_3",
        "Sham_oxy_4",
        "Sham_Sal_1",
        "Sham_Sal_2",
        "Sham_Sal_3",
        "Sham_Sal_4",
        "SNI_oxy_1",
        "SNI_oxy_2",
        "SNI_oxy_3",
        "SNI_oxy_4",
        "SNI_Sal_1",
        "SNI_Sal_2",
        "SNI_Sal_3",
        "SNI_Sal_4"
    ],
    "Condition": [
        "Sham_oxy",
        "Sham_oxy",
        "Sham_oxy",
        "Sham_oxy",
        "Sham_Sal",
        "Sham_Sal",
        "Sham_Sal",
        "Sham_Sal",
        "SNI_oxy",
        "SNI_oxy",
        "SNI_oxy",
        "SNI_oxy",
        "SNI_Sal",
        "SNI_Sal",
        "SNI_Sal",
        "SNI_Sal"
    ]
})

samplesheet.to_csv("samplesheet.csv", index=False)

samplesheet

,sample,Condition
0,Sham_oxy_1,Sham_oxy
1,Sham_oxy_2,Sham_oxy
2,Sham_oxy_3,Sham_oxy
3,Sham_oxy_4,Sham_oxy
4,Sham_Sal_1,Sham_Sal
5,Sham_Sal_2,Sham_Sal
6,Sham_Sal_3,Sham_Sal
7,Sham_Sal_4,Sham_Sal
8,SNI_oxy_1,SNI_oxy
9,SNI_oxy_2,SNI_oxy


Please paste here the command you used. You may need to inspect the provided expression matrix more closely and create additional files, like a samplesheet (based on the column names) or a contrast file (there happens to also be one in `data/` that you can use).

In [ ]:
!nextflow run nf-core/differentialabundance \
    -r 2.0.0
    -profile docker \
    --input samplesheet.csv \
    --contrasts data/contrasts.csv \
    --matrix data/salmon.merged.gene_counts.tsv \
    # genome number couldd be collected from rersult from last step.
    --outdir diffab_result


 N E X T F L O W   ~  version 26.04.6

Launching `https://github.com/nf-core/differentialabundance` [deadly_rutherford] revision: 5cb5f9858f [master]

WARN: Unrecognized config option 'validation.defaultIgnoreParams'
WARN: Unrecognized config option 'validation.monochromeLogs'

------------------------------------------------------
                                        ,--./,-.
        ___     __   __   __   ___     /,-._.--~'
  |\ | |__  __ /  ` /  \ |__) |__         }  {
  | \| |       \__, \__/ |  \ |___     \`-._,-`-,
                                        `._,._,'
  nf-core/differentialabundance 2.0.0
------------------------------------------------------

[base] Input/output options
  input                       : samplesheet.csv
  contrasts                   : data/contrasts.csv
  outdir                      : diffab_result

[base] Abundance values
  matrix                      : data/salmon.merged.gene_counts.tsv

[preprocessing] Filtering
  filtering_min_abundance     : 1


Explain all the parameters you set and why you set them in this way. If you used or created additional files as input, explain what they are used for.

    -r 2.0.0 # control the version, insure reproducibility
    -profile docker \ # container 
   


What were the outputs of the pipeline?

Differential analysis result, statistical plots of expressing data by treatment groups.

Would you exclude any samples? If yes, which and why?

SNI_Sal_2 and SNI_Sal_4. 
Shifted the whole analysis, different from the other members of the group in the , from PCA plot.

How many genes were differentially expressed in each contrast? Does this confirm what the paper mentions?

Sham_oxy versus Sham_Sal in Condition: 7 up and 0 down.\
SNI_oxy versus SNI_Sal in Condition: 1 up and 17 down.\

This doesn't matchh the table, the paper reports hundreds of gaps per region, a lot more than our results.

The paper mentions differentially expressed genes in three brain regions : the NAc, mPFC and VTA. Briefly explain what these 3 regions are.

VTA, ventral tegmental area: a mid brain region containing dopamine neurnos of the reward system.
NAc, nucleus accumbens: It received dopamine frm the VTA and input from mPFC, intergrates reward, motivation and dreiinforcement learning.
mPFC, medial prefronotal cortex: cortial region for decisino -making, impulse control and emotion regulation.

Is there anyway from the paper and the material and methods for us to know which genes are included in these regions?

There is no direct information in the paper. 

Once you have your list of differentially expressed genes, do you think just communicating those to the biologists would be sufficient? What does the publication state?

Not sufficient.

Please reproduce the Venn Diagram from Figure 3, not taking into account the brain regions but just the contrasts mentionned.

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for matplotlib-venn: filename=matplotlib_venn-1.1.2-py3-none-any.whl size=45388 sha256=13fca948fe7f78b7afc3f4dfd643d1819bff5182236925e4eca2e3a07fca3a4a
  Stored in directory: /Users/ivan/Library/Caches/pip/wheels/d1/5f/e6/771479559f992b8398265ebf61f8a3d33ca0b8f75552e06ad2
Successfully built matplotlib-venn
[]


KeyError: 'Sham-Oxy vs Sham-Sal'